# House Price Prediction — Exploratory Data Analysis & Model Training

This notebook demonstrates the full ML workflow for the House Price Prediction project:

1. Load the dataset
2. Exploratory Data Analysis (EDA)
3. Visualizations
4. Preprocessing
5. Model training
6. Evaluation

> Note: This notebook is for **exploration and demonstration**. The actual
> production training happens in `train_model.py`, and the app uses the
> model saved by that script.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")
%matplotlib inline

## 2. Load the Dataset

Make sure you have already run `python generate_dataset.py` from the project root.

In [ ]:
df = pd.read_csv("../data/house_prices.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of the target variable: price
plt.figure(figsize=(8, 5))
sns.histplot(df["price"], kde=True, bins=40, color="steelblue")
plt.title("Distribution of House Prices (Lakhs)")
plt.xlabel("Price (Lakhs)")
plt.ylabel("Count")
plt.show()

In [ ]:
# Relationship between area and price
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="area_sqft", y="price", hue="location", alpha=0.6)
plt.title("Area vs Price (colored by Location)")
plt.xlabel("Area (sqft)")
plt.ylabel("Price (Lakhs)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Average price by location
plt.figure(figsize=(8, 5))
avg_price_by_location = df.groupby("location")["price"].mean().sort_values(ascending=False)
sns.barplot(x=avg_price_by_location.index, y=avg_price_by_location.values, palette="viridis")
plt.title("Average Price by Location")
plt.ylabel("Average Price (Lakhs)")
plt.xticks(rotation=20)
plt.show()

In [ ]:
# Correlation heatmap for numeric features
numeric_cols = ["area_sqft", "bedrooms", "bathrooms", "parking", "house_age", "price"]
plt.figure(figsize=(7, 5))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Effect of house age on price
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="house_age", y="price", alpha=0.5, color="darkorange")
plt.title("House Age vs Price")
plt.xlabel("House Age (years)")
plt.ylabel("Price (Lakhs)")
plt.show()

## 4. Preprocessing

In [ ]:
NUMERIC_FEATURES = ["area_sqft", "bedrooms", "bathrooms", "parking", "house_age"]
CATEGORICAL_FEATURES = ["location"]
TARGET = "price"

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## 5. Model Training

In [ ]:
# Linear Regression
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)

# Random Forest Regressor
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])
rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)

print("Both models trained successfully.")

## 6. Evaluation

In [ ]:
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE={mae:.2f}, RMSE={rmse:.2f}, R2={r2:.4f}")
    return mae, rmse, r2

evaluate(y_test, lr_preds, "Linear Regression")
evaluate(y_test, rf_preds, "Random Forest Regressor")

In [ ]:
# Visualize predicted vs actual prices (Random Forest)
plt.figure(figsize=(7, 7))
plt.scatter(y_test, rf_preds, alpha=0.5, color="teal")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual Price (Lakhs)")
plt.ylabel("Predicted Price (Lakhs)")
plt.title("Random Forest: Actual vs Predicted Price")
plt.show()

## Conclusion

Both Linear Regression and Random Forest Regressor were trained on the
synthetic dataset. The Random Forest model generally captures the
non-linear relationships (like depreciation with house age) slightly
better, which is reflected in its evaluation metrics above.

The final production model used by the Streamlit app is selected and
saved automatically by `train_model.py` (whichever model scores higher
on R² is chosen).